In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')

import os
import tensorflow as tf
import keras
import cv2
import glob
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model
from tensorflow.keras import layers , models, optimizers

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.applications import ResNet50V2

# **Data Generator without data augmentation**

In [2]:
# specifing new image shape for resnet
img_shape = 224
batch_size = 64
train_data_path = './train/'
test_data_path = './test/'

In [3]:
train_preprocessor = ImageDataGenerator(
        rescale = 1 / 255.,
        validation_split=0.25) # set validation split

test_preprocessor = ImageDataGenerator(
    rescale = 1 / 255.,
)

train_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode='rgb',
    batch_size=batch_size,
    subset='training', 
)


val_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode='rgb',
    batch_size=batch_size,
    subset='validation',
) # set as validation data


test_data = test_preprocessor.flow_from_directory(
    test_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

Found 21535 images belonging to 7 classes.
Found 7175 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


# **Fine-Tuning ResNet50V2**

In [4]:
ResNet50V2 = tf.keras.applications.ResNet50V2(input_shape=(224, 224, 3),
                                               include_top= False,
                                               weights='imagenet'
                                               )

In [5]:
# Freezing all layers except last 50
ResNet50V2.trainable = True

for layer in ResNet50V2.layers[:-50]:
    layer.trainable = False

In [6]:
def Create_ResNet50V2_Model():

    model = Sequential([
                      ResNet50V2,
                      Dropout(0.25),
                      BatchNormalization(),
                      Flatten(),
                      Dense(64, activation='relu'),
                      BatchNormalization(),
                      Dropout(0.5),
                      Dense(7,activation='softmax')
                    ])
    return model

In [7]:
ResNet50V2_Model = Create_ResNet50V2_Model()

ResNet50V2_Model.summary()

ResNet50V2_Model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50v2 (Functional)     (None, 7, 7, 2048)        23564800  
                                                                 
 dropout (Dropout)           (None, 7, 7, 2048)        0         
                                                                 
 batch_normalization (Batch  (None, 7, 7, 2048)        8192      
 Normalization)                                                  
                                                                 
 flatten (Flatten)           (None, 100352)            0         
                                                                 
 dense (Dense)               (None, 64)                6422592   
                                                                 
 batch_normalization_1 (Bat  (None, 64)                256       
 chNormalization)                                       

**Specifying Callbacks**

In [9]:
# Create Callback Checkpoint
checkpoint_path = "ResNet50V2_Model_non_d_aug.h5"

Checkpoint = ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True)

# Create Early Stopping Callback to monitor the accuracy
Early_Stopping = EarlyStopping(monitor = 'val_accuracy', patience = 7, restore_best_weights = True, verbose=1)

# Create ReduceLROnPlateau Callback to reduce overfitting by decreasing learning
Reducing_LR = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                  factor=0.2,
                                                  patience=2,
#                                                   min_lr=0.00005,
                                                  verbose=1)

callbacks = [Checkpoint, Early_Stopping, Reducing_LR]

steps_per_epoch = train_data.n // train_data.batch_size
validation_steps = val_data.n // val_data.batch_size

In [10]:
history = ResNet50V2_Model.fit(train_data,
                               validation_data = val_data , 
                               epochs=20, 
                               batch_size=batch_size,
                               callbacks = callbacks, 
                               steps_per_epoch=steps_per_epoch, 
                               validation_steps=validation_steps
                              )

Epoch 1/20
336/336 [==============================] - ETA: 0s - loss: 1.4268 - accuracy: 0.5102

/opt/homebrew/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


336/336 [==============================] - 835s 2s/step - loss: 1.4268 - accuracy: 0.5102 - val_loss: 1.3839 - val_accuracy: 0.4972 - lr: 0.0010
Epoch 2/20
336/336 [==============================] - 725s 2s/step - loss: 1.0923 - accuracy: 0.6161 - val_loss: 1.1779 - val_accuracy: 0.5624 - lr: 0.0010
Epoch 3/20
336/336 [==============================] - 720s 2s/step - loss: 0.9379 - accuracy: 0.6729 - val_loss: 1.5744 - val_accuracy: 0.5594 - lr: 0.0010
Epoch 4/20
336/336 [==============================] - ETA: 0s - loss: 0.8014 - accuracy: 0.7230
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
336/336 [==============================] - 724s 2s/step - loss: 0.8014 - accuracy: 0.7230 - val_loss: 1.4671 - val_accuracy: 0.5820 - lr: 0.0010
Epoch 5/20
336/336 [==============================] - 726s 2s/step - loss: 0.4133 - accuracy: 0.8755 - val_loss: 0.9996 - val_accuracy: 0.6508 - lr: 2.0000e-04
Epoch 6/20
336/336 [==============================] - 725s 2s/ste

KeyboardInterrupt: 

# **Evaluating ResNet50V2**

In [11]:
ResNet50V2_Score = ResNet50V2_Model.evaluate(test_data)

print("    Test Loss: {:.5f}".format(ResNet50V2_Score[0]))
print("Test Accuracy: {:.2f}%".format(ResNet50V2_Score[1] * 100))

113/113 [==============================] - 187s 2s/step - loss: 1.1358 - accuracy: 0.6532
    Test Loss: 1.13576
Test Accuracy: 65.32%
